In [16]:
import os
import numpy as np
import pandas as pd
from readlif.reader import LifFile
from skimage import filters, morphology, measure
from skimage.segmentation import clear_border

############ Parameters
LIF_PATH     = "/abyss/dlafonta/imaging/DNA_FISH/DMSO_568/DMSO_all_curated_568_clean.lif"
OUTPUT_DIR   = "/sharehome/dlafonta/imaging/DMSO_568"
OUTPUT_CSV   = "DMSO_568_foci_SON_intensity.csv"
COMBINED_CSV = "DMSO_568_combined_measurements.csv"

GAUSSIAN_SIGMA   = 2
DILATION_PX      = 5
CH1_MIN_SIZE     = 100
CH1_CLEAR_BORDER = True
CH3_PERCENTILE   = 97
CH3_MIN_SIZE     = 37
CH3_MAX_SIZE     = 170
CH3_CLEAR_BORDER = False

############# Helper functions
def norm01(img):
    """Min-max scale to 0-1."""
    mn, mx = img.min(), img.max()
    return (img - mn) / (mx - mn) if mx > mn else img.astype(float)

def segment_ch1(image):
    """DAPI nuclear mask: Otsu on smoothed image, size/hole filter, clear border."""
    smooth = filters.gaussian(image, sigma=GAUSSIAN_SIGMA)
    mask   = smooth > filters.threshold_otsu(smooth)
    mask   = morphology.remove_small_objects(mask, min_size=CH1_MIN_SIZE)
    mask   = morphology.remove_small_holes(mask, area_threshold=CH1_MIN_SIZE)
    if CH1_CLEAR_BORDER:
        mask = clear_border(mask)
    return mask

def segment_ch3_in_ch1(ch3_image, ch1_mask):
    """Foci mask: percentile threshold over in-nucleus pixels, size-filtered."""
    values_in_ch1 = ch3_image[ch1_mask]
    if values_in_ch1.size == 0:
        return np.zeros(ch3_image.shape, dtype=bool)
    smooth = filters.gaussian(ch3_image, sigma=GAUSSIAN_SIGMA)
    thr    = np.percentile(values_in_ch1, CH3_PERCENTILE)
    mask   = (smooth > thr) & ch1_mask
    mask   = morphology.remove_small_objects(mask, min_size=CH3_MIN_SIZE)
    if CH3_MAX_SIZE is not None:
        labeled = measure.label(mask)
        for prop in measure.regionprops(labeled):
            if prop.area > CH3_MAX_SIZE:
                mask[labeled == prop.label] = False
    if CH3_CLEAR_BORDER:
        mask = clear_border(mask)
    return mask

def measure_foci(ch3_mask, ch2_raw, coll_idx, z, coll_name, mean_intensity):
    """Mean raw Ch2 in each focus, its ring, and combined. One dict per focus."""
    labeled = measure.label(ch3_mask)
    disk    = morphology.disk(DILATION_PX)
    rows    = []
    for prop in measure.regionprops(labeled):
        foci = ch3_mask & (labeled == prop.label)
        ring = morphology.binary_dilation(foci, disk) & ~foci
        both = foci | ring
        rows.append({
            "collection_idx"  : coll_idx,
            "collection_name" : coll_name,
            "slice_identifier": f"{coll_idx}_{z}",
            "foci_id"         : prop.label,
            "foci_area_px"    : prop.area,
            "mean_SON_foci"   : ch2_raw[foci].mean() if foci.any() else np.nan,
            "mean_SON_ring"   : ch2_raw[ring].mean() if ring.any() else np.nan,
            "mean_SON_all"    : ch2_raw[both].mean() if both.any() else np.nan,
            "mean_intensity"  : mean_intensity,
        })
    return rows

########### Run analysis
lif      = LifFile(LIF_PATH)
all_rows = []
for coll_idx in range(len(lif.image_list)):
    img = lif.get_image(coll_idx)
    for z in range(img.dims.z):
        ch1_raw = np.array(img.get_frame(z=z, t=0, c=0)).astype(float)
        ch2_raw = np.array(img.get_frame(z=z, t=0, c=1)).astype(float)
        ch3_raw = np.array(img.get_frame(z=z, t=0, c=2)).astype(float)
        ch1_mask = segment_ch1(norm01(ch1_raw))
        if ch1_mask.sum() == 0:
            continue
        # Overall SON signal: mean raw Ch2 within the nuclear mask (per slice)
        mean_intensity = ch2_raw[ch1_mask].mean()
        ch3_mask = segment_ch3_in_ch1(norm01(ch3_raw), ch1_mask)
        all_rows.extend(measure_foci(ch3_mask, ch2_raw, coll_idx, z, img.name, mean_intensity))

df = pd.DataFrame(all_rows, columns=[
    "collection_idx", "collection_name", "slice_identifier", "foci_id",
    "foci_area_px", "mean_SON_foci", "mean_SON_ring", "mean_SON_all", "mean_intensity"
])
os.makedirs(OUTPUT_DIR, exist_ok=True)
df.to_csv(os.path.join(OUTPUT_DIR, OUTPUT_CSV), index=False)
df.to_csv(os.path.join(OUTPUT_DIR, COMBINED_CSV), index=False)

In [17]:
import os
import numpy as np
import pandas as pd
from readlif.reader import LifFile
from skimage import filters, morphology, measure
from skimage.segmentation import clear_border

############ Parameters
LIF_PATH     = "/abyss/dlafonta/imaging/DNA_FISH/TD_568/TD_all_curated_568_clean.lif"
OUTPUT_DIR   = "/sharehome/dlafonta/imaging/TD_568"
OUTPUT_CSV   = "TD_568_foci_SON_intensity.csv"
COMBINED_CSV = "TD_568_combined_measurements.csv"

GAUSSIAN_SIGMA   = 2
DILATION_PX      = 5
CH1_MIN_SIZE     = 100
CH1_CLEAR_BORDER = True
CH3_PERCENTILE   = 97
CH3_MIN_SIZE     = 37
CH3_MAX_SIZE     = 170
CH3_CLEAR_BORDER = False

############# Helper functions
def norm01(img):
    """Min-max scale to 0-1."""
    mn, mx = img.min(), img.max()
    return (img - mn) / (mx - mn) if mx > mn else img.astype(float)

def segment_ch1(image):
    """DAPI nuclear mask: Otsu on smoothed image, size/hole filter, clear border."""
    smooth = filters.gaussian(image, sigma=GAUSSIAN_SIGMA)
    mask   = smooth > filters.threshold_otsu(smooth)
    mask   = morphology.remove_small_objects(mask, min_size=CH1_MIN_SIZE)
    mask   = morphology.remove_small_holes(mask, area_threshold=CH1_MIN_SIZE)
    if CH1_CLEAR_BORDER:
        mask = clear_border(mask)
    return mask

def segment_ch3_in_ch1(ch3_image, ch1_mask):
    """Foci mask: percentile threshold over in-nucleus pixels, size-filtered."""
    values_in_ch1 = ch3_image[ch1_mask]
    if values_in_ch1.size == 0:
        return np.zeros(ch3_image.shape, dtype=bool)
    smooth = filters.gaussian(ch3_image, sigma=GAUSSIAN_SIGMA)
    thr    = np.percentile(values_in_ch1, CH3_PERCENTILE)
    mask   = (smooth > thr) & ch1_mask
    mask   = morphology.remove_small_objects(mask, min_size=CH3_MIN_SIZE)
    if CH3_MAX_SIZE is not None:
        labeled = measure.label(mask)
        for prop in measure.regionprops(labeled):
            if prop.area > CH3_MAX_SIZE:
                mask[labeled == prop.label] = False
    if CH3_CLEAR_BORDER:
        mask = clear_border(mask)
    return mask

def measure_foci(ch3_mask, ch2_raw, coll_idx, z, coll_name, mean_intensity):
    """Mean raw Ch2 in each focus, its ring, and combined. One dict per focus."""
    labeled = measure.label(ch3_mask)
    disk    = morphology.disk(DILATION_PX)
    rows    = []
    for prop in measure.regionprops(labeled):
        foci = ch3_mask & (labeled == prop.label)
        ring = morphology.binary_dilation(foci, disk) & ~foci
        both = foci | ring
        rows.append({
            "collection_idx"  : coll_idx,
            "collection_name" : coll_name,
            "slice_identifier": f"{coll_idx}_{z}",
            "foci_id"         : prop.label,
            "foci_area_px"    : prop.area,
            "mean_SON_foci"   : ch2_raw[foci].mean() if foci.any() else np.nan,
            "mean_SON_ring"   : ch2_raw[ring].mean() if ring.any() else np.nan,
            "mean_SON_all"    : ch2_raw[both].mean() if both.any() else np.nan,
            "mean_intensity"  : mean_intensity,
        })
    return rows

########### Run analysis
lif      = LifFile(LIF_PATH)
all_rows = []
for coll_idx in range(len(lif.image_list)):
    img = lif.get_image(coll_idx)
    for z in range(img.dims.z):
        ch1_raw = np.array(img.get_frame(z=z, t=0, c=0)).astype(float)
        ch2_raw = np.array(img.get_frame(z=z, t=0, c=1)).astype(float)
        ch3_raw = np.array(img.get_frame(z=z, t=0, c=2)).astype(float)
        ch1_mask = segment_ch1(norm01(ch1_raw))
        if ch1_mask.sum() == 0:
            continue
        # Overall SON signal: mean raw Ch2 within the nuclear mask (per slice)
        mean_intensity = ch2_raw[ch1_mask].mean()
        ch3_mask = segment_ch3_in_ch1(norm01(ch3_raw), ch1_mask)
        all_rows.extend(measure_foci(ch3_mask, ch2_raw, coll_idx, z, img.name, mean_intensity))

df = pd.DataFrame(all_rows, columns=[
    "collection_idx", "collection_name", "slice_identifier", "foci_id",
    "foci_area_px", "mean_SON_foci", "mean_SON_ring", "mean_SON_all", "mean_intensity"
])
os.makedirs(OUTPUT_DIR, exist_ok=True)
df.to_csv(os.path.join(OUTPUT_DIR, OUTPUT_CSV), index=False)
df.to_csv(os.path.join(OUTPUT_DIR, COMBINED_CSV), index=False)

In [18]:
import os
import numpy as np
import pandas as pd
from readlif.reader import LifFile
from skimage import filters, morphology, measure
from skimage.segmentation import clear_border

############ Parameters
LIF_PATH     = "/abyss/dlafonta/imaging/DNA_FISH/DMSO_647/DMSO_all_curated_647_cleaner.lif"
OUTPUT_DIR   = "/sharehome/dlafonta/imaging/DMSO_647"
OUTPUT_CSV   = "DMSO_647_foci_SON_intensity.csv"
COMBINED_CSV = "DMSO_647_combined_measurements.csv"

GAUSSIAN_SIGMA   = 2
DILATION_PX      = 5
CH1_MIN_SIZE     = 100
CH1_CLEAR_BORDER = True
CH3_PERCENTILE   = 98
CH3_MIN_SIZE     = 34
CH3_MAX_SIZE     = None
CH3_CLEAR_BORDER = False

############# Helper functions
def norm01(img):
    """Min-max scale to 0-1."""
    mn, mx = img.min(), img.max()
    return (img - mn) / (mx - mn) if mx > mn else img.astype(float)

def segment_ch1(image):
    """DAPI nuclear mask: Otsu on smoothed image, size/hole filter, clear border."""
    smooth = filters.gaussian(image, sigma=GAUSSIAN_SIGMA)
    mask   = smooth > filters.threshold_otsu(smooth)
    mask   = morphology.remove_small_objects(mask, min_size=CH1_MIN_SIZE)
    mask   = morphology.remove_small_holes(mask, area_threshold=CH1_MIN_SIZE)
    if CH1_CLEAR_BORDER:
        mask = clear_border(mask)
    return mask

def segment_ch3_in_ch1(ch3_image, ch1_mask):
    """Foci mask: percentile threshold over in-nucleus pixels, size-filtered."""
    values_in_ch1 = ch3_image[ch1_mask]
    if values_in_ch1.size == 0:
        return np.zeros(ch3_image.shape, dtype=bool)
    smooth = filters.gaussian(ch3_image, sigma=GAUSSIAN_SIGMA)
    thr    = np.percentile(values_in_ch1, CH3_PERCENTILE)
    mask   = (smooth > thr) & ch1_mask
    mask   = morphology.remove_small_objects(mask, min_size=CH3_MIN_SIZE)
    if CH3_MAX_SIZE is not None:
        labeled = measure.label(mask)
        for prop in measure.regionprops(labeled):
            if prop.area > CH3_MAX_SIZE:
                mask[labeled == prop.label] = False
    if CH3_CLEAR_BORDER:
        mask = clear_border(mask)
    return mask

def measure_foci(ch3_mask, ch2_raw, coll_idx, z, coll_name, mean_intensity):
    """Mean raw Ch2 in each focus, its ring, and combined. One dict per focus."""
    labeled = measure.label(ch3_mask)
    disk    = morphology.disk(DILATION_PX)
    rows    = []
    for prop in measure.regionprops(labeled):
        foci = ch3_mask & (labeled == prop.label)
        ring = morphology.binary_dilation(foci, disk) & ~foci
        both = foci | ring
        rows.append({
            "collection_idx"  : coll_idx,
            "collection_name" : coll_name,
            "slice_identifier": f"{coll_idx}_{z}",
            "foci_id"         : prop.label,
            "foci_area_px"    : prop.area,
            "mean_SON_foci"   : ch2_raw[foci].mean() if foci.any() else np.nan,
            "mean_SON_ring"   : ch2_raw[ring].mean() if ring.any() else np.nan,
            "mean_SON_all"    : ch2_raw[both].mean() if both.any() else np.nan,
            "mean_intensity"  : mean_intensity,
        })
    return rows

########### Run analysis
lif      = LifFile(LIF_PATH)
all_rows = []
for coll_idx in range(len(lif.image_list)):
    img = lif.get_image(coll_idx)
    for z in range(img.dims.z):
        ch1_raw = np.array(img.get_frame(z=z, t=0, c=0)).astype(float)
        ch2_raw = np.array(img.get_frame(z=z, t=0, c=1)).astype(float)
        ch3_raw = np.array(img.get_frame(z=z, t=0, c=2)).astype(float)
        ch1_mask = segment_ch1(norm01(ch1_raw))
        if ch1_mask.sum() == 0:
            continue
        # Overall SON signal: mean raw Ch2 within the nuclear mask (per slice)
        mean_intensity = ch2_raw[ch1_mask].mean()
        ch3_mask = segment_ch3_in_ch1(norm01(ch3_raw), ch1_mask)
        all_rows.extend(measure_foci(ch3_mask, ch2_raw, coll_idx, z, img.name, mean_intensity))

df = pd.DataFrame(all_rows, columns=[
    "collection_idx", "collection_name", "slice_identifier", "foci_id",
    "foci_area_px", "mean_SON_foci", "mean_SON_ring", "mean_SON_all", "mean_intensity"
])
os.makedirs(OUTPUT_DIR, exist_ok=True)
df.to_csv(os.path.join(OUTPUT_DIR, OUTPUT_CSV), index=False)
df.to_csv(os.path.join(OUTPUT_DIR, COMBINED_CSV), index=False)

In [19]:
import os
import numpy as np
import pandas as pd
from readlif.reader import LifFile
from skimage import filters, morphology, measure
from skimage.segmentation import clear_border

############ Parameters
LIF_PATH     = "/abyss/dlafonta/imaging/DNA_FISH/TD_647/TD_all_curated_647_cleaner.lif"
OUTPUT_DIR   = "/sharehome/dlafonta/imaging/TD_647"
OUTPUT_CSV   = "TD_647_foci_SON_intensity.csv"
COMBINED_CSV = "TD_647_combined_measurements.csv"

GAUSSIAN_SIGMA   = 2
DILATION_PX      = 5
CH1_MIN_SIZE     = 100
CH1_CLEAR_BORDER = True
CH3_PERCENTILE   = 98
CH3_MIN_SIZE     = 34
CH3_MAX_SIZE     = None
CH3_CLEAR_BORDER = False

############# Helper functions
def norm01(img):
    """Min-max scale to 0-1."""
    mn, mx = img.min(), img.max()
    return (img - mn) / (mx - mn) if mx > mn else img.astype(float)

def segment_ch1(image):
    """DAPI nuclear mask: Otsu on smoothed image, size/hole filter, clear border."""
    smooth = filters.gaussian(image, sigma=GAUSSIAN_SIGMA)
    mask   = smooth > filters.threshold_otsu(smooth)
    mask   = morphology.remove_small_objects(mask, min_size=CH1_MIN_SIZE)
    mask   = morphology.remove_small_holes(mask, area_threshold=CH1_MIN_SIZE)
    if CH1_CLEAR_BORDER:
        mask = clear_border(mask)
    return mask

def segment_ch3_in_ch1(ch3_image, ch1_mask):
    """Foci mask: percentile threshold over in-nucleus pixels, size-filtered."""
    values_in_ch1 = ch3_image[ch1_mask]
    if values_in_ch1.size == 0:
        return np.zeros(ch3_image.shape, dtype=bool)
    smooth = filters.gaussian(ch3_image, sigma=GAUSSIAN_SIGMA)
    thr    = np.percentile(values_in_ch1, CH3_PERCENTILE)
    mask   = (smooth > thr) & ch1_mask
    mask   = morphology.remove_small_objects(mask, min_size=CH3_MIN_SIZE)
    if CH3_MAX_SIZE is not None:
        labeled = measure.label(mask)
        for prop in measure.regionprops(labeled):
            if prop.area > CH3_MAX_SIZE:
                mask[labeled == prop.label] = False
    if CH3_CLEAR_BORDER:
        mask = clear_border(mask)
    return mask

def measure_foci(ch3_mask, ch2_raw, coll_idx, z, coll_name, mean_intensity):
    """Mean raw Ch2 in each focus, its ring, and combined. One dict per focus."""
    labeled = measure.label(ch3_mask)
    disk    = morphology.disk(DILATION_PX)
    rows    = []
    for prop in measure.regionprops(labeled):
        foci = ch3_mask & (labeled == prop.label)
        ring = morphology.binary_dilation(foci, disk) & ~foci
        both = foci | ring
        rows.append({
            "collection_idx"  : coll_idx,
            "collection_name" : coll_name,
            "slice_identifier": f"{coll_idx}_{z}",
            "foci_id"         : prop.label,
            "foci_area_px"    : prop.area,
            "mean_SON_foci"   : ch2_raw[foci].mean() if foci.any() else np.nan,
            "mean_SON_ring"   : ch2_raw[ring].mean() if ring.any() else np.nan,
            "mean_SON_all"    : ch2_raw[both].mean() if both.any() else np.nan,
            "mean_intensity"  : mean_intensity,
        })
    return rows

########### Run analysis
lif      = LifFile(LIF_PATH)
all_rows = []
for coll_idx in range(len(lif.image_list)):
    img = lif.get_image(coll_idx)
    for z in range(img.dims.z):
        ch1_raw = np.array(img.get_frame(z=z, t=0, c=0)).astype(float)
        ch2_raw = np.array(img.get_frame(z=z, t=0, c=1)).astype(float)
        ch3_raw = np.array(img.get_frame(z=z, t=0, c=2)).astype(float)
        ch1_mask = segment_ch1(norm01(ch1_raw))
        if ch1_mask.sum() == 0:
            continue
        # Overall SON signal: mean raw Ch2 within the nuclear mask (per slice)
        mean_intensity = ch2_raw[ch1_mask].mean()
        ch3_mask = segment_ch3_in_ch1(norm01(ch3_raw), ch1_mask)
        all_rows.extend(measure_foci(ch3_mask, ch2_raw, coll_idx, z, img.name, mean_intensity))

df = pd.DataFrame(all_rows, columns=[
    "collection_idx", "collection_name", "slice_identifier", "foci_id",
    "foci_area_px", "mean_SON_foci", "mean_SON_ring", "mean_SON_all", "mean_intensity"
])
os.makedirs(OUTPUT_DIR, exist_ok=True)
df.to_csv(os.path.join(OUTPUT_DIR, OUTPUT_CSV), index=False)
df.to_csv(os.path.join(OUTPUT_DIR, COMBINED_CSV), index=False)